<a href="https://colab.research.google.com/github/CRUM25/Examen-C11-Problema-de-optimizacion-de-rutas-en-grafos-/blob/main/Codigo_fuente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Examen C11 (Problema de optimización de rutas en grafos)
---
**Candidato: Carlos Rodrigo Uruchurtu Morales**

#Objetivo del proyecto

El siguiente caso de negocio se desarrollaro tomando en cuenta el ambiente de **GCP(Google Cloude Function)** peros no es extricto se puede migrar ya sea AWS o cualquier otro servicio

Se esta diseñando un Data Pipeline en GCP que ingesta eventos en tiempo real y los almacena para análisis en BigQuery.Las aristas representan la latencia del procesamiento en segundos de transferir y procesar la data entre servicios.

La data contiene PII(Información Personal), por politicas de cumplimiento y sguridad, es estrictamente obligatorio que la data pase por Cloud Dataflow(u) hacia Cloud DLP (v) para enmascarar los datos antes de que puedan ser guardados en bases de datos analíticas.

Diccionario
* o: Nodo Origen (API)
* a: Cloud Functions
* u: Dataflow
* v: Cloud DLP
* c: Cloud Storage
* d: Nodo Final (Bigquery)
* $x_n$: costo (latencia)


#Librerias

In [1]:
!pip install pyvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 38.8 MB/s eta 0:00:00


In [2]:
from pyvis.network import Network
import IPython

from scipy.sparse.csgraph import dijkstra
import networkx as nx
import scipy.sparse as sp
import pandas as pd
import numpy as np
import time
import random

#Visualización de la conexión(grafo dataset 1)

In [3]:
G=nx.DiGraph()
G.add_node(1, label="o",shape="ellipse",level=0)
G.add_node(2, label="a",shape="ellipse",level=1)
G.add_node(3, label="u",shape="ellipse",level=2)
G.add_node(4, label="v",shape="ellipse",level=3)
G.add_node(5, label="c",shape="ellipse",level=4)
G.add_node(6, label="d",shape="ellipse",level=5)


G.add_edge(1, 2,label="x1")
G.add_edge(1, 3,label="x2")
G.add_edge(2, 3,label="x3")
G.add_edge(3, 4,value=3,label="x4")
G.add_edge(4, 5,label="x5")
G.add_edge(2, 6,label="x6")
G.add_edge(4, 6,label="x7")
G.add_edge(5, 6,label="x8")


In [4]:
grafo= Network(notebook=True, directed=True, height="500px", cdn_resources='remote')
grafo.from_nx(G)
grafo.force_atlas_2based()
grafo.show("Conexión_entre_sistemas.html")
IPython.display.HTML(filename="Conexión_entre_sistemas.html")


Conexión_entre_sistemas.html


#Creación del algoritmo

In [5]:
class model:
  def __init__(self, nodo_des:int, nodos:list, nodo_ori: int = 0, nodo_obligado:list | None=None, arist_prohib:list[list[int]] | None=None):
    self.nodo_ori=nodo_ori
    self.nodo_des=nodo_des
    self.nodos=nodos
    self.nodo_obligado=nodo_obligado
    self.arist_prohib= arist_prohib if arist_prohib else []

  def obtener_ruta(self, predecesores, origen, destino):
        ruta = []
        nodo_actual = destino
        while nodo_actual != origen:
            if nodo_actual < 0:
                return []
            ruta.insert(0, str(self.nodos[nodo_actual]))
            nodo_actual = predecesores[nodo_actual]
        ruta.insert(0, str(self.nodos[origen]))
        return ruta

  def preparar_matriz(self, matriz_ori):

        matriz = np.copy(matriz_ori)
        for u, v in self.arist_prohib:
            matriz[u, v] = 0.0

        epsilon = 1e-5
        matriz[matriz > 0] += epsilon
        return matriz

  def calcular_distancia_min(self,matriz_ori):

    matriz = self.preparar_matriz(matriz_ori)

    if self.nodo_obligado is not None:
      try:
        distancias_desde_o, predecesores_o=dijkstra(csgraph=matriz,directed=True,indices=self.nodo_ori,return_predecessors=True)
        costo_1=distancias_desde_o[self.nodo_obligado[0]]

        if np.isinf(costo_1):
            raise ValueError("No hay ruta válida desde el nodo origen hasta el nodo obligatorio.")
        ruta_1=self.obtener_ruta(predecesores_o,self.nodo_ori,self.nodo_obligado[0])

        costo_uv=matriz[self.nodo_obligado[0],self.nodo_obligado[1]]
        if costo_uv==0:
          raise ValueError("La arista obligatoria no existe.")

        distancias_desde_v, predecesores_v=dijkstra(csgraph=matriz, directed=True, indices=self.nodo_obligado[1], return_predecessors=True)
        costo_2=distancias_desde_v[self.nodo_des]

        if np.isinf(costo_2):
          raise ValueError("No hay ruta válida desde 'v' hasta 'd'.")
        ruta_2=self.obtener_ruta(predecesores_v,self.nodo_obligado[1] , self.nodo_des)

        costo_total=costo_1 + costo_uv + costo_2
        ruta_completa = ruta_1 + ruta_2

        print(f"Latencia: {round(costo_total, 2)} segundos")
        print(f"Ruta a seguir: {' -> '.join(ruta_completa)}")

      except ValueError as e:
        print(f"Error: {e}")

    else:
        distancias_ori, predecesores_ori=dijkstra(csgraph=matriz, directed=True, indices=self.nodo_ori, return_predecessors=True)
        costo=distancias_ori[self.nodo_des]

        if np.isinf(costo):
            print("No hay ruta válida desde el nodo origen hasta el nodo destino.")
        else:
            ruta = self.obtener_ruta(predecesores_ori, self.nodo_ori, self.nodo_des)
            print(f"Latencia: {round(costo, 2)} segundos")
            print(f"Ruta a seguir: {' -> '.join(ruta)}")



#Generación de datos

##Dataset 1

### Costos

In [6]:
np.random.seed(431)
xn=np.random.randint(1, 21, size=8)

### Nodos y aristas

In [7]:
nodos = ['o', 'a', 'u', 'v', 'c', 'd']

idx = {nodo: i for i, nodo in enumerate(nodos)}

matriz = np.zeros((len(nodos),len(nodos)))
matriz[idx['o'], idx['a']] = xn[0]
matriz[idx['o'], idx['u']] = xn[1]
matriz[idx['a'], idx['u']] = xn[2]
matriz[idx['u'], idx['v']] = xn[3]
matriz[idx['v'], idx['c']] = xn[4]
matriz[idx['a'], idx['d']] = xn[5]
matriz[idx['v'], idx['d']] = xn[6]
matriz[idx['c'], idx['d']] = xn[7]

arista_obligado=[2,3]

##Dataset 2

In [8]:
num_nodos = 3000
densidad = 0.4

nombres= [f"Nodo_{i}" for i in range(num_nodos)]

matriz_aleatoria = sp.random(num_nodos, num_nodos, density=densidad, format='csr',random_state=431,data_rvs=lambda s: np.random.randint(1, 100, size=s))

matriz_dataset = matriz_aleatoria.toarray()

ori = 0
des = 2999
u_ob = 1270
v_ob = 1360

matriz_dataset[u_ob, v_ob] = 15.0
prohibidas = [[0, des], [1000, des], [0, 2000]]

print(f"Dataset generado: {num_nodos} nodos y aproximadamente {int(num_nodos * num_nodos * densidad)} aristas.")

Dataset generado: 3000 nodos y aproximadamente 3600000 aristas.


#Casos

## 1.Caso con solución

En este caso es la solución al problema base obligatorio

In [9]:
alg=model(nodo_des=5,nodos=nodos,nodo_obligado=arista_obligado)

In [10]:
alg.calcular_distancia_min(matriz)

Latencia: 43.0 segundos
Ruta a seguir: o -> u -> v -> d


## 2.Caso sin solución

En este caso rompemos el flujo definido en el grafo yendo de $d \rightarrow o$

In [11]:
alg=model(nodo_des=0,nodos=nodos,nodo_ori=5,nodo_obligado=arista_obligado)
alg.calcular_distancia_min(matriz)

Error: No hay ruta válida desde el nodo origen hasta el nodo obligatorio.


## 3.Caso borde relevantes

En este caso se define una arista que no cumple con el flujo del Data Pipeline donde pedimos que Dataflow mande data a la API

In [12]:
arista_obligado=[3,0]

In [13]:
alg=model(nodo_des=5,nodos=nodos,nodo_obligado=arista_obligado)

In [14]:
alg.calcular_distancia_min(matriz)

Error: La arista obligatoria no existe.


En este caso suponemos que se interrumpe el flujo por cualquier cosa y solamente tenemos el nodo origen

In [15]:
alg=model(nodo_des=0,nodos=nodos)

In [16]:
alg.calcular_distancia_min(matriz)

Latencia: 0.0 segundos
Ruta a seguir: o


##4.Escalabilidad, estrés computacional o repetición
de consultas.

In [17]:
model2 = model(nodo_des=des,nodos=nombres,nodo_ori=ori,nodo_obligado=[u_ob, v_ob],arist_prohib=prohibidas)

start_time = time.time()

model2.calcular_distancia_min(matriz_dataset)
end_time = time.time()
print(f"\nTiempo de ejecución: {round((end_time - start_time) * 1000, 2)} ms")

Latencia: 22.0 segundos
Ruta a seguir: Nodo_0 -> Nodo_866 -> Nodo_1787 -> Nodo_2156 -> Nodo_1270 -> Nodo_1360 -> Nodo_1124 -> Nodo_2440 -> Nodo_2999

Tiempo de ejecución: 2699.89 ms


In [18]:
model2 = model(nodo_des=des,nodos=nombres,nodo_ori=ori)

start_time = time.time()

model2.calcular_distancia_min(matriz_dataset)
end_time = time.time()
print(f"\nTiempo de ejecución: {round((end_time - start_time) * 1000, 2)} ms")

Latencia: 4.0 segundos
Ruta a seguir: Nodo_0 -> Nodo_2179 -> Nodo_2440 -> Nodo_2999

Tiempo de ejecución: 1464.74 ms
